In [8]:
from pathlib import Path

from omegaconf import OmegaConf
from openai import OpenAI

config_path = next(
    (path for path in (Path.cwd() / "env.yaml", Path.cwd().parent / "env.yaml") if path.exists()),
    None,
)
if config_path is None:
    raise FileNotFoundError("Could not find env.yaml in the current directory or its parent")

config = OmegaConf.to_container(OmegaConf.load(config_path), resolve=True)
api_key = (
    config.get("anthropic_api_key")
    or config.get("policy_api_key")
    or config.get("API_KEY")
    or config.get("api_key")
)
if not api_key:
    raise ValueError(f"No API key found in {config_path}")

base_url = "https://inference-api.nvidia.com/v1"
base_url = base_url.rstrip("/").removesuffix("/chat/completions").removesuffix("/messages")

client = OpenAI(
    api_key=api_key,
    base_url=base_url,
)

response = client.chat.completions.create(
    model=config.get("anthropic_model_name") or config.get("model") or config.get("policy_model_name") or "us/aws/anthropic/eccn-claude-opus-4-8",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant.",
        },
        {
            "role": "user",
            "content": "Explain how to plan a safe software release in three concise steps.",
        },
    ],
    max_tokens=1024,
)

print(response.choices[0].message.content)

# Planning a Safe Software Release in 3 Steps

## 1. Prepare and Validate
- **Freeze the code** and ensure all features are complete and merged
- **Test thoroughly** across unit, integration, and staging environments that mirror production
- **Create a rollback plan** and back up your data and current stable version

## 2. Deploy Gradually
- **Use a phased rollout** (e.g., canary or blue-green deployment) to expose the release to a small subset of users first
- **Monitor key metrics** (error rates, latency, resource usage) in real time
- **Schedule wisely**—deploy during low-traffic windows with your team available

## 3. Monitor and Respond
- **Watch dashboards and alerts** closely for the first hours after release
- **Be ready to roll back** quickly if critical issues appear
- **Document outcomes** and gather feedback to improve your next release

---

**Quick tip:** Automate as much as possible (testing, deployment, monitoring) to reduce human error and speed up recovery.


In [26]:
import httpx

try:
    config
    api_key
except NameError:
    from pathlib import Path

    from omegaconf import OmegaConf

    config_path = next(
        (path for path in (Path.cwd() / "env.yaml", Path.cwd().parent / "env.yaml") if path.exists()),
        None,
    )
    if config_path is None:
        raise FileNotFoundError("Could not find env.yaml in the current directory or its parent")

    config = OmegaConf.to_container(OmegaConf.load(config_path), resolve=True)
    api_key = config.get("anthropic_api_key")
    if not api_key:
        raise ValueError(f"No API key found in {config_path}")

model = "us/aws/anthropic/eccn-claude-opus-4-8"

# Equivalent to the chat/completions curl above, but on the native Messages endpoint.
# System prompt moves to top-level "system"; only user turns go in "messages".
response = httpx.post(
    "https://inference-api.nvidia.com/v1/messages",
    headers={
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}",
        "anthropic-version": "2023-06-01",
    },
    json={
        "model": model,
        "max_tokens": 1024,
        "system": "Today's date is 2024-06-01.",
        "messages": [
            {
                "role": "user",
                "content": "Explain how to plan a safe software release in three concise steps.",
            }
        ],
        "temperature": 1,
        # "thinking": {
        #     "type": "adaptive",
        # },
        "tools": [
            {
                "name": "get_weather",
                "description": "Get the current weather for a location.",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "location": {"type": "string"},
                        "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                    },
                    "required": ["location"],
                },
            }
        ],
        "tool_choice": {"type": "auto"},  # optional; default behavior
        # "top_k": 5,
        # "top_p": 0.7,
    },
    timeout=600,
)
response.raise_for_status()
result = response.json()

for block in result.get("content", []):
    if block.get("type") == "text":
        print(block["text"])
    elif block.get("type") == "tool_use":
        print(f"Tool call: {block['name']}({block['input']})")
        # Execute the tool, then send results back in a follow-up request:
        # messages.append({"role": "assistant", "content": result["content"]})
        # messages.append({
        #     "role": "user",
        #     "content": [{
        #         "type": "tool_result",
        #         "tool_use_id": block["id"],
        #         "content": "72°F and sunny",
        #     }],
        # })

Here's how to plan a safe software release in three concise steps:

**1. Prepare and Test Thoroughly**
Freeze the code, run your full test suite (unit, integration, and regression tests), and validate the release in a staging environment that mirrors production. Confirm all dependencies, configurations, and database migrations are accounted for.

**2. Deploy Gradually with Safeguards**
Roll out incrementally—using techniques like canary releases or blue-green deployment—so you expose the change to a small subset of users first. Keep a rollback plan ready and ensure feature flags or kill switches are in place to disable problematic features quickly.

**3. Monitor and Verify**
Watch key metrics (error rates, latency, resource usage) and logs in real time immediately after release. Confirm the release is healthy before expanding to all users, and be prepared to roll back instantly if anomalies appear.
